# Task 1: News Topic Classifier Using BERT

## Objective
Fine-tune BERT on AG News and evaluate using Accuracy and F1-Score.

## 1. Install Dependencies

In [ ]:
!pip install transformers datasets evaluate accelerate scikit-learn gradio -q

## 2. Load Dataset

In [ ]:
from datasets import load_dataset
ag_news=load_dataset('ag_news')
ag_news

## 3. Tokenization

In [ ]:
from transformers import AutoTokenizer
model_name='bert-base-uncased'
tokenizer=AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch['text'],padding='max_length',truncation=True,max_length=128)

tokenized=ag_news.map(tokenize,batched=True)

## 4. Model Training

In [ ]:
from transformers import AutoModelForSequenceClassification,TrainingArguments,Trainer
import evaluate, numpy as np

model=AutoModelForSequenceClassification.from_pretrained(model_name,num_labels=4)
acc=evaluate.load('accuracy')
f1=evaluate.load('f1')

def metrics(eval_pred):
    logits,labels=eval_pred
    preds=np.argmax(logits,axis=-1)
    return {'accuracy':acc.compute(predictions=preds,references=labels)['accuracy'],
            'f1':f1.compute(predictions=preds,references=labels,average='weighted')['f1']}

args=TrainingArguments(output_dir='bert_news',evaluation_strategy='epoch',num_train_epochs=2,per_device_train_batch_size=16)
trainer=Trainer(model=model,args=args,train_dataset=tokenized['train'].shuffle(seed=42).select(range(10000)),eval_dataset=tokenized['test'].select(range(2000)),compute_metrics=metrics)
trainer.train()

## 5. Final Evaluation

In [ ]:
trainer.evaluate()

## 6. Gradio Deployment

In [ ]:
import gradio as gr
from transformers import pipeline
clf=pipeline('text-classification',model=model,tokenizer=tokenizer)
labels=['World','Sports','Business','Sci/Tech']

def predict(text):
    r=clf(text)[0]
    return r

gr.Interface(fn=predict,inputs='textbox',outputs='label').launch()

## Final Summary
BERT fine-tuned on AG News with Accuracy and F1 evaluation and Gradio deployment.